# Datas e fusos com Python

Use datetime, timedelta e zoneinfo. Nas datas usadas aqui, São Paulo está em UTC−3 e Cuiabá em UTC−4. Use os nomes das zonas para aplicar as regras do fuso, em vez de subtrair horas manualmente.

Use este notebook antes da versão com pandas. Usamos a biblioteca padrão do Python (3.9 ou superior). Os dados de fusos IANA podem exigir o pacote tzdata em alguns ambientes.

**Como estudar:** execute as demos de cima para baixo. Antes de cada exercício, escreva entradas, saída esperada e passos. Preencha a célula vazia, execute e compare com exemplos. Consulte o gabarito só depois da tentativa.

Uma função precisa ser definida antes de ser chamada. Se aparecer `NameError`, confira o nome e execute novamente a célula da definição. Ao reiniciar o kernel, execute as definições outra vez.

## Roteiro da professora — aula integrada de 60 minutos

Os quatro notebooks pertencem a **uma aula de aproximadamente uma hora no total**. Execute as células necessárias de cima para baixo; a agenda seleciona demonstrações, não todos os exercícios. Cada exercício tem uma resolução logo após a célula de tentativa, com resultado visível quando necessário. Os gabaritos e oficinas originais continuam disponíveis para estudo posterior.

| Tempo | Conteúdo | Material |
|---|---|---|
| 0–5 min | Texto, data e problema de negócio | Datas básico |
| 5–12 min | Ler, formatar e somar datas | Datas básico |
| 12–22 min | Fuso, UTC e mudança de dia | Datas básico |
| 22–27 min | Converter uma coluna e localizar | Datas nível 2 |
| 27–32 min | O que é regex e leitura do padrão | Regex básico |
| 32–42 min | Procurar, validar e limpar | Regex básico |
| 42–49 min | Reutilizar a regra em função | Regex básico |
| 49–55 min | Aplicar padrões a uma coluna | Regex nível 2 |
| 55–60 min | Verificação e fechamento | Perguntas do roteiro |

**Preparação:** abra os quatro arquivos antes da aula e selecione o kernel Python. Os níveis 2 precisam de pandas e IPython; a demonstração com `format="ISO8601"` requer pandas 2.0 ou superior. Teste os imports previamente. A biblioteca `zoneinfo` precisa dos dados IANA; se o ambiente não os encontrar, instale `tzdata` no kernel. Os básicos usam a biblioteca padrão, mas podem precisar desses dados de fusos.

Se houver atraso, priorize os exemplos básicos e mostre apenas as tabelas dos níveis 2, preservando os cinco minutos finais de verificação.


### Fala e leitura guiada — 5–12 minutos
“`"05/10/2026"` começa como texto. Para calcular prazos, precisamos interpretá-lo como uma data.”
br - dd/mm/aaaa

- `from ... import ...` traz ferramentas da biblioteca padrão.
- `datetime` representa data e hora; `timedelta` representa um intervalo.
- `strptime(texto, formato)` **lê** texto: `%d` é dia, `%m` mês e `%Y` ano com quatro dígitos. As barras são literais.
- `strftime(formato)` **produz** texto a partir de uma data. Pense: entrada versus saída.
- `timedelta(days=7)` representa sete dias; somá-lo à data calcula um prazo.

O ponto em `data.strftime(...)` acessa um método do objeto. O conteúdo entre parênteses são os argumentos. Peça que prevejam a data após sete dias: **12/10/2026**. Não some 7 ao texto nem ao número do dia manualmente.

BR = dd/mm/aaaa
USA = mm/dd/aaaa

## Demo — Texto vira datetime

strptime lê um texto segundo um formato. %d é dia, %m é mês e %Y é ano com quatro dígitos.

In [8]:
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo

data_ini = "05/10/2026" ## string em formato dd/mm/aaaa
print(type(data_ini))

#data = datetime.strptime("05/10/2026", "%d/%m/%Y")
data = datetime.strptime(data_ini, "%d/%m/%Y")
print(type(data)) # <class 'datetime.datetime'>
print(data.strftime("%Y-%m-%d"))
print(data + timedelta(days=7))

<class 'str'>
<class 'datetime.datetime'>
2026-10-05
2026-10-12 00:00:00


### O que é um fuso horário? — explicação para falar
“Fuso horário é uma regra que uma região usa para definir sua hora local. A Terra gira, por isso o Sol não aparece no mesmo momento em todos os lugares. As regiões adotam horários por convenções e decisões políticas; não são apenas faixas perfeitamente desenhadas no mapa.”

**UTC** é uma referência mundial de tempo. **UTC−3** significa que, naquele instante, o relógio local marca três horas a menos que UTC. **UTC−4** significa quatro horas a menos. Uma zona como `America/Sao_Paulo` guarda regras de deslocamento ao longo do tempo, inclusive mudanças históricas. Um deslocamento como `-03:00` informa apenas a diferença indicada naquela data.

Nos exemplos deste notebook, em 05/10/2026:

| Referência/local | Hora do mesmo instante |
|---|---|
| UTC | 13:00 |
| São Paulo | 10:00 |
| Cuiabá | 09:00 |

“Não aconteceram três eventos. É **um evento visto em três relógios**. Converter o fuso não altera o instante.”

**Pergunta para a turma:** se uma compra ocorreu às 02:30 UTC de 06/10/2026, qual é a data em São Paulo? **Resposta:** 05/10/2026 às 23:30. Por isso convertemos para o fuso do negócio **antes** de contar vendas por dia.

Uma hora como `09:00` sem cidade, deslocamento ou informação de origem não define um instante único. Não adivinhe o fuso. O horário de verão e mudanças nas regras são motivos para usar zonas nomeadas em vez de subtrair horas manualmente.

### Leia o código com a turma — 12–22 minutos
`fromisoformat` interpreta ano, mês, dia, hora e `+00:00`, que indica UTC. `T` é o separador comum entre data e hora no formato ISO. `ZoneInfo(...)` identifica a zona desejada; `astimezone(...)` converte o instante para ela. A saída mostra também o deslocamento.

Execute esta demonstração e depois a célula de entrada do **exercício 4**. Resolva ao vivo com `converter_fuso(madrugada, "America/Sao_Paulo")`, após executar a definição da função. A turma deve olhar o dia além da hora.

## Demo — UTC vira horário local

fromisoformat interpreta o deslocamento +00:00. astimezone preserva o instante e muda a representação local. Uma data sem fuso não informa qual horário local representa.

In [9]:
instante = datetime.fromisoformat("2026-10-05T13:00:00+00:00")
print(instante.astimezone(ZoneInfo("America/Sao_Paulo")))
print(instante.astimezone(ZoneInfo("America/Cuiaba")))

2026-10-05 10:00:00-03:00
2026-10-05 09:00:00-04:00


### Por que uma função?
“Se tenho cem horários, não quero copiar cem vezes a regra. Dou um nome à transformação.” `def` define a função; `texto` e `destino` são parâmetros. Uma chamada fornece argumentos concretos. `return` entrega o resultado a quem chamou. O `if` recusa uma entrada sem fuso; `raise ValueError` explica o problema. Mostre a função agora e deixe a criação de funções mais longas para a prática posterior.

## Demo — Repetição vira função

Contrato: entrada ISO com deslocamento explícito; saída datetime no destino. Recusamos datas sem fuso, pois não sabemos se representam UTC.

In [11]:
def converter_fuso(texto, destino):
    momento = datetime.fromisoformat(texto)
    if momento.tzinfo is None:
        raise ValueError("Informe o deslocamento do fuso na entrada.")
    return momento.astimezone(ZoneInfo(destino))

print(converter_fuso("2026-10-05T13:00:00+00:00", "America/Cuiaba"))

2026-10-05 09:00:00-04:00


## Exercício 1

Converta os três textos para datetime. Primeiro faça três chamadas; depois crie `ler_data(texto)` e use um for. Compare os resultados.

In [13]:
textos = ["05/10/2026", "06/10/2026", "07/10/2026"]
textos

['05/10/2026', '06/10/2026', '07/10/2026']

In [18]:
# Escreva sua solução aqui.
antes = [datetime.strptime(t, "%d/%m/%Y") for t in textos]
antes

def ler_data(texto):
    return datetime.strptime(texto, "%d/%m/%Y")

depois = []

for texto in textos:
    aux = ler_data(texto)
    depois.append(aux)

depois

[datetime.datetime(2026, 10, 5, 0, 0),
 datetime.datetime(2026, 10, 6, 0, 0),
 datetime.datetime(2026, 10, 7, 0, 0)]

### Resolução para mostrar na aula

### Solução 1

Execute a célula abaixo depois das entradas do exercício. Leia os comentários e compare o resultado com a regra pedida. Os `assert` verificam o resultado esperado; quando passam, não mostram mensagem.

In [19]:
antes = [datetime.strptime(t, "%d/%m/%Y") for t in textos]

def ler_data(texto):
    return datetime.strptime(texto, "%d/%m/%Y")

depois = []
for texto in textos:
    depois.append(ler_data(texto))
#assert antes == depois
print(depois)

[datetime.datetime(2026, 10, 5, 0, 0), datetime.datetime(2026, 10, 6, 0, 0), datetime.datetime(2026, 10, 7, 0, 0)]


## Exercício 2

Crie `somar_dias(texto, quantidade)` usando ler_data. Retorne uma string no formato dia/mês/ano. Teste virada de mês e ano.

In [ ]:
# Escreva sua solução aqui.


### Resolução para mostrar na aula

### Solução 2

Execute a célula abaixo depois das entradas do exercício. Leia os comentários e compare o resultado com a regra pedida. Os `assert` verificam o resultado esperado; quando passam, não mostram mensagem.

In [25]:
def somar_dias(data_como_texto, quantidade):
    data_como_data = datetime.strptime(data_como_texto, "%d/%m/%Y")
    print(f'data como data {data_como_data}')
    data_final = data_como_data + timedelta(days=quantidade)
    print(f'data final {data_final}')
    return data_final.strftime("%d/%m/%Y")

print(somar_dias("31/10/2026", 1))

data como data 2026-10-31 00:00:00
data final 2026-11-01 00:00:00
01/11/2026


In [26]:
def somar_dias(texto, quantidade):
    return (datetime.strptime(texto, "%d/%m/%Y") + timedelta(days=quantidade)).strftime("%d/%m/%Y")

print(somar_dias("31/10/2026", 1))


01/11/2026


In [27]:
def somar_dias(texto, quantidade):
    return (ler_data(texto) + timedelta(days=quantidade)).strftime("%d/%m/%Y")

print(somar_dias("31/10/2026", 1))

01/11/2026


## Exercício 3

Converta cada horário UTC para São Paulo e Cuiabá chamando converter_fuso. Guarde os resultados em duas listas. Qual é a diferença de horário local?

In [31]:
eventos = ["2026-10-05T13:00:00+00:00", "2026-10-05T00:30:00+00:00"]

In [ ]:
# Escreva sua solução aqui.


### Resolução para mostrar na aula

### Solução 3

Execute a célula abaixo depois das entradas do exercício. Leia os comentários e compare o resultado com a regra pedida. Os `assert` verificam o resultado esperado; quando passam, não mostram mensagem.

In [32]:
sp = [converter_fuso(t, "America/Sao_Paulo") for t in eventos]
cuiaba = [converter_fuso(t, "America/Cuiaba") for t in eventos]

# Cuiabá mostra uma hora a menos nessas datas; os instantes são iguais.

print("São Paulo:", sp)
print("Cuiabá:", cuiaba)

São Paulo: [datetime.datetime(2026, 10, 5, 10, 0, tzinfo=zoneinfo.ZoneInfo(key='America/Sao_Paulo')), datetime.datetime(2026, 10, 4, 21, 30, tzinfo=zoneinfo.ZoneInfo(key='America/Sao_Paulo'))]
Cuiabá: [datetime.datetime(2026, 10, 5, 9, 0, tzinfo=zoneinfo.ZoneInfo(key='America/Cuiaba')), datetime.datetime(2026, 10, 4, 20, 30, tzinfo=zoneinfo.ZoneInfo(key='America/Cuiaba'))]


## Exercício 4

Converta este instante para as duas cidades. Confira a data, não apenas a hora. Explique por que a data local muda.

In [33]:
madrugada = "2026-10-06T02:30:00+00:00"

In [ ]:
# Escreva sua solução aqui.


### Resolução para mostrar na aula

### Solução 4

Execute a célula abaixo depois das entradas do exercício. Leia os comentários e compare o resultado com a regra pedida. Os `assert` verificam o resultado esperado; quando passam, não mostram mensagem.

In [34]:
sp_noite = converter_fuso(madrugada, "America/Sao_Paulo")
cuiaba_noite = converter_fuso(madrugada, "America/Cuiaba")
assert sp_noite.strftime("%d/%m/%Y %H:%M") == "05/10/2026 23:30"
assert cuiaba_noite.strftime("%d/%m/%Y %H:%M") == "05/10/2026 22:30"
# O mesmo instante ainda cai no dia anterior no horário local.
print(sp_noite)
print(cuiaba_noite)

2026-10-05 23:30:00-03:00
2026-10-05 22:30:00-04:00


## Exercício 5

Crie `em_expediente(horario, inicio=8, fim=18)`. Receba um datetime já convertido para a cidade e retorne bool. Considere início incluído e fim excluído. Teste 07:59, 08:00 e 18:00.

In [ ]:
# Escreva sua solução aqui.


### Resolução para mostrar na aula

### Solução 5

Execute a célula abaixo depois das entradas do exercício. Leia os comentários e compare o resultado com a regra pedida. Os `assert` verificam o resultado esperado; quando passam, não mostram mensagem.

In [36]:
def em_expediente(horario, inicio=8, fim=18):
    return inicio <= horario.hour < fim


for texto, esperado in [("2026-10-05T10:59:00+00:00", False), ("2026-10-05T11:00:00+00:00", True), ("2026-10-05T21:00:00+00:00", False)]:
   # assert em_expediente(converter_fuso(texto, "America/Sao_Paulo")) == esperado
    print(f'{texto} está no expediente:', em_expediente(converter_fuso(texto, "America/Sao_Paulo")))

2026-10-05T10:59:00+00:00 está no expediente: False
2026-10-05T11:00:00+00:00 está no expediente: True
2026-10-05T21:00:00+00:00 está no expediente: False


## Exercício 6

Crie `resumir_evento(texto, cidade)` que chama converter_fuso e em_expediente, retornando um dicionário com horário formatado e indicador de expediente. Teste também a rejeição de uma entrada sem fuso.

In [ ]:
# Escreva sua solução aqui.


### Resolução para mostrar na aula

### Solução 6

Execute a célula abaixo depois das entradas do exercício. Leia os comentários e compare o resultado com a regra pedida. Os `assert` verificam o resultado esperado; quando passam, não mostram mensagem.

In [37]:
def resumir_evento(texto, cidade):
    local = converter_fuso(texto, cidade)
    return {"horario": local.strftime("%d/%m/%Y %H:%M"), "expediente": em_expediente(local)}
#assert resumir_evento(eventos[0], "America/Cuiaba") == {"horario": "05/10/2026 09:00", "expediente": True}
try:
    converter_fuso("2026-10-05T13:00:00", "America/Sao_Paulo")
except ValueError as erro:
    print(erro)
else:
    raise AssertionError("Era necessário rejeitar a entrada sem fuso.")
print(resumir_evento(eventos[0], "America/Cuiaba"))

Informe o deslocamento do fuso na entrada.
{'horario': '05/10/2026 09:00', 'expediente': True}


## Gabaritos comentados

Pare aqui se ainda estiver praticando. Compare as regras e os resultados, não apenas a aparência do código.

### Solução 1

In [ ]:
antes = [datetime.strptime(t, "%d/%m/%Y") for t in textos]
def ler_data(texto):
    return datetime.strptime(texto, "%d/%m/%Y")
depois = []
for texto in textos:
    depois.append(ler_data(texto))
assert antes == depois

### Solução 2

In [ ]:
def somar_dias(texto, quantidade):
    return (ler_data(texto) + timedelta(days=quantidade)).strftime("%d/%m/%Y")
assert somar_dias("31/10/2026", 1) == "01/11/2026"
assert somar_dias("31/12/2026", 1) == "01/01/2027"

### Solução 3

In [ ]:
sp = [converter_fuso(t, "America/Sao_Paulo") for t in eventos]
cuiaba = [converter_fuso(t, "America/Cuiaba") for t in eventos]
assert [d.hour for d in sp] == [10, 15]
assert [d.hour for d in cuiaba] == [9, 14]
# Cuiabá mostra uma hora a menos nessas datas; os instantes são iguais.
assert sp[0] == cuiaba[0]

### Solução 4

In [ ]:
sp_noite = converter_fuso(madrugada, "America/Sao_Paulo")
cuiaba_noite = converter_fuso(madrugada, "America/Cuiaba")
assert sp_noite.strftime("%d/%m/%Y %H:%M") == "05/10/2026 23:30"
assert cuiaba_noite.strftime("%d/%m/%Y %H:%M") == "05/10/2026 22:30"
# O mesmo instante ainda cai no dia anterior no horário local.

### Solução 5

In [ ]:
def em_expediente(horario, inicio=8, fim=18):
    return inicio <= horario.hour < fim
for texto, esperado in [("2026-10-05T10:59:00+00:00", False), ("2026-10-05T11:00:00+00:00", True), ("2026-10-05T21:00:00+00:00", False)]:
    assert em_expediente(converter_fuso(texto, "America/Sao_Paulo")) == esperado

### Solução 6

In [ ]:
def resumir_evento(texto, cidade):
    local = converter_fuso(texto, cidade)
    return {"horario": local.strftime("%d/%m/%Y %H:%M"), "expediente": em_expediente(local)}
assert resumir_evento(eventos[0], "America/Cuiaba") == {"horario": "05/10/2026 09:00", "expediente": True}
try:
    converter_fuso("2026-10-05T13:00:00", "America/Sao_Paulo")
except ValueError as erro:
    print(erro)
else:
    raise AssertionError("Era necessário rejeitar a entrada sem fuso.")